In [ ]:
import tensorflow as tf
import numpy as np
import scipy.io
import os
import matplotlib.pyplot as plt
import pandas as pd

patch_size = 8
epoch = 50
batch_size = 32
learning_rate_base = 0.01

print('Loading Data and Generating Training/Testing Samples...')

DATA_PATH = "/content/drive/MyDrive/model"

mat_data = scipy.io.loadmat(os.path.join(DATA_PATH, 'hydice_data.mat'))
HR_MSI = mat_data['msi_data']
HR_HSI = mat_data['hydhyper_data']

HR_MSI = np.transpose(HR_MSI, (1, 2, 0))
HR_HSI = np.transpose(HR_HSI, (1, 2, 0))

print(f"HR_MSI Shape After Fix: {HR_MSI.shape}")
print(f"HR_HSI Shape After Fix: {HR_HSI.shape}")

Height_HR, Width_HR, Band_MSI = HR_MSI.shape
Band_HSI = HR_HSI.shape[2]

msi_max = np.max(HR_MSI)
HR_MSI = HR_MSI.astype(np.float32) / msi_max
HR_HSI = HR_HSI.astype(np.float32) / msi_max

split_index = int(Height_HR * 0.7)
Training_MSI = HR_MSI[:split_index, :, :]
Training_HSI = HR_HSI[:split_index, :, :]
Testing_MSI  = HR_MSI[split_index:, :, :]
Testing_HSI  = HR_HSI[split_index:, :, :]

print(f"Training Data Shape: {Training_MSI.shape}, {Training_HSI.shape}")
print(f"Testing Data Shape: {Testing_MSI.shape}, {Testing_HSI.shape}")

def extract_patches(data, patch_size):
    patches = []
    for i in range(0, data.shape[0] - patch_size + 1, patch_size):
        for j in range(0, data.shape[1] - patch_size + 1, patch_size):
            patch = data[i:i + patch_size, j:j + patch_size, :]
            patches.append(patch)
    return patches

Train_MSI_Patches = extract_patches(Training_MSI, patch_size)
Train_HSI_Patches = extract_patches(Training_HSI, patch_size)
Test_MSI_Patches  = extract_patches(Testing_MSI,  patch_size)
Test_HSI_Patches  = extract_patches(Testing_HSI,  patch_size)

print(f"Train MSI Shape: {np.array(Train_MSI_Patches).shape}, Train HSI Shape: {np.array(Train_HSI_Patches).shape}")
print(f"Test MSI Shape: {np.array(Test_MSI_Patches).shape}, Test HSI Shape: {np.array(Test_HSI_Patches).shape}")

class ECALayer(tf.keras.layers.Layer):
    def __init__(self, k_size=5, **kwargs):
        super(ECALayer, self).__init__(**kwargs)
        self.k_size = k_size
        self.global_avg_pool = tf.keras.layers.GlobalAveragePooling2D(keepdims=True)
        self.conv1d = tf.keras.layers.Conv1D(filters=1, kernel_size=k_size, padding="same", activation="sigmoid", use_bias=False)

    def call(self, inputs):
        gap = self.global_avg_pool(inputs)
        gap_reshaped = tf.keras.layers.Reshape((-1, 1))(gap)
        attention = self.conv1d(gap_reshaped)
        attention = tf.keras.layers.Reshape((1, -1, 1))(attention)
        return inputs * attention


def build_model(input_shape, output_channels):
    inputs = tf.keras.Input(shape=input_shape)

    x = tf.keras.layers.Conv2D(64, (1, 1), activation='relu', padding='same')(inputs)

    shortcut = x

    x1 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x2 = tf.keras.layers.Conv2D(64, (1, 1), activation='relu', padding='same')(x)
    x  = tf.keras.layers.Concatenate()([x1, x2])
    x  = tf.keras.layers.Conv2D(64, (1, 1), activation='relu', padding='same')(x)
    x  = ECALayer(k_size=5)(x)
    x  = tf.keras.layers.Add()([x, shortcut])

    shortcut = x
    x1 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x2 = tf.keras.layers.Conv2D(64, (1, 1), activation='relu', padding='same')(x)
    x  = tf.keras.layers.Concatenate()([x1, x2])
    x  = tf.keras.layers.Conv2D(64, (1, 1), activation='relu', padding='same')(x)
    x  = ECALayer(k_size=5)(x)
    x  = tf.keras.layers.Add()([x, shortcut])

    x1 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x2 = tf.keras.layers.Conv2D(64, (1, 1), activation='relu', padding='same')(x)
    x  = tf.keras.layers.Concatenate()([x1, x2])
    x  = tf.keras.layers.Conv2D(64, (1, 1), activation='relu', padding='same')(x)
    x  = ECALayer(k_size=5)(x)
    x  = tf.keras.layers.Add()([x, shortcut])

    outputs = tf.keras.layers.Conv2D(output_channels, (1, 1), activation=None, padding='same')(x)
    outputs = ECALayer(k_size=5)(outputs)

    model = tf.keras.Model(inputs, outputs)
    return model

input_shape     = (patch_size, patch_size, Band_MSI)
output_channels = Band_HSI

model = build_model(input_shape, output_channels)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate_base), loss='mae')

model.fit(Train_MSI_Patches, Train_HSI_Patches,
          batch_size=batch_size, epochs=epoch,
          validation_data=(Test_MSI_Patches, Test_HSI_Patches))

model.save("/content/drive/MyDrive/hyperiondata/trained_hyperspectral_model_hydice.h5")

print("Model Training Complete & Saved!")